In [1]:
import duckdb
import pandas as pd

from IPython.display import display
pd.options.display.max_columns = None
pd.options.display.max_rows = 30
pd.get_option("display.max_rows")

import os
from pathlib import Path

raw_data_path = Path("/Volumes/SANDISKUSBC/MLOps/hdb_price_prediction_mlops/data/raw")
sorted(os.listdir(raw_data_path))

# Create a process data directory if it doesn't already exist
processed_data_path = Path("/Volumes/SANDISKUSBC/MLOps/hdb_price_prediction_mlops/data/processed")
processed_data_path.mkdir(parents=True, exist_ok=True)

In [ ]:
train_df= pd.read_csv(raw_data_path/'clean_train.csv')
eval_df= pd.read_csv(raw_data_path/'clean_eval.csv')
hold_df= pd.read_csv(raw_data_path/'hold.csv')

#source data: https://tablebuilder.singstat.gov.sg/table/TS/M212161
resale_index_df = pd.read_csv(raw_data_path/'hdb_resale_index.csv')

In [3]:
train_df.head()

,month,town,blk_no,road_name,building,postal,resale_price,storey_range,flat_type,flat_model,lease_commence_date,remaining_lease_years,remaining_lease_months,floor_area_sqm,floor_area_sqft,price_per_sqft,planning_area_ura,region_ura,x,y,latitude,longitude,closest_mrt_station,distance_to_mrt_meters,transport_type,line_color,distance_to_cbd,closest_pri_school,distance_to_pri_school_meters
0,2022-03-01,CHOA CHU KANG,816A,KEAT HONG LINK,KEAT HONG MIRAGE,681816,550000.0,19 TO 21,4 ROOM,Model A,2017-09-01,94,6,92.0,990.2788,555.399146,CHOA CHU KANG,WEST REGION,18596.867822,39780.256257,1.376032,103.748825,Keat Hong,285.529337,LRT,Grey,15288.837561,SOUTH VIEW PRIMARY SCHOOL,684.627817
1,2022-04-01,CHOA CHU KANG,205,CHOA CHU KANG CENTRAL,NIL,680205,522888.0,01 TO 03,5 ROOM,Improved,1989-05-01,66,0,122.0,1313.1958,398.179769,CHOA CHU KANG,WEST REGION,18567.392344,40321.553471,1.380927,103.748560,Keat Hong,262.790995,LRT,Grey,15675.388451,SOUTH VIEW PRIMARY SCHOOL,266.995332
2,2022-04-01,CHOA CHU KANG,818C,CHOA CHU KANG AVENUE 1,KEAT HONG MIRAGE,683818,595000.0,01 TO 03,5 ROOM,Improved,2017-09-01,94,6,112.0,1205.5568,493.547878,CHOA CHU KANG,WEST REGION,18521.801436,39856.591264,1.376722,103.748150,Keat Hong,231.162515,LRT,Grey,15395.662063,SOUTH VIEW PRIMARY SCHOOL,585.204929
3,2022-04-01,CHOA CHU KANG,7,TECK WHYE AVENUE,TECK WHYE HEIGHTS I,680007,645000.0,22 TO 24,5 ROOM,Model A,1984-03-01,60,11,132.0,1420.8348,453.958476,CHOA CHU KANG,WEST REGION,19230.883013,40382.086654,1.381474,103.754522,Phoenix,499.411641,LRT,Grey,15240.120250,TECK WHYE PRIMARY SCHOOL,276.758965
4,2022-05-01,CHOA CHU KANG,111,TECK WHYE LANE,NIL,680111,420000.0,01 TO 03,4 ROOM,Model A,1989-09-01,66,5,103.0,1108.6817,378.828297,CHOA CHU KANG,WEST REGION,18961.791376,39984.533910,1.377879,103.752104,Teck Whye,222.457650,LRT,Grey,15157.140648,TECK WHYE PRIMARY SCHOOL,651.676254


In [4]:
resale_index_df.tail()

,quarter,resale_price_index
139,2024 4Q,197.9
140,2025 1Q,201.0
141,2025 2Q,202.9
142,2025 3Q,203.7
143,2025 4Q,203.6


### Feature Engineer/Encoding
* left join hdb_resale_index 
* map flat_type to rank by median resale_price
* map region_ura to rank by median resale_price
* map town to rank by median resale_price
* storey_range_rank
* flat_model_rank_map by median resale_price

In [5]:
def derive_quarter_column(df, date_col='month'):
    """
    Derives a new 'quarter' column from a date column to match the 'YYYY nQ' format.
    Example: '2022-03-01' -> '2022 1Q'
    """
    # Convert the column to datetime objects
    dt_col = pd.to_datetime(df[date_col])
    
    # Extract the year and quarter, and format as "YYYY nQ"
    df['quarter'] = dt_col.dt.year.astype(str) + ' ' + dt_col.dt.quarter.astype(str) + 'Q'
    
    return df

# Apply the function to all of your data splits
train_df = derive_quarter_column(train_df)
eval_df = derive_quarter_column(eval_df)
hold_df = derive_quarter_column(hold_df)

# Verify the output
train_df[['month', 'quarter']].head()




,month,quarter
0,2022-03-01,2022 1Q
1,2022-04-01,2022 2Q
2,2022-04-01,2022 2Q
3,2022-04-01,2022 2Q
4,2022-05-01,2022 2Q


In [6]:
# 1. First, load the resale index data
#resale_index_df = pd.read_csv(raw_data_path / 'hdb_resale_index.csv')

# 2. Define the left join function
def join_resale_index(df, index_df):
    """
    Left joins the resale_price_index from index_df to the main df 
    using the 'quarter' column.
    """
    # Perform a left merge on the 'quarter' column
    # Only taking the necessary columns from index_df to avoid duplication
    merged_df = pd.merge(
        df, 
        index_df[['quarter', 'resale_price_index']], 
        on='quarter', 
        how='left'
    )
    
    return merged_df

# 3. Apply the function to all of your data splits
train_df = join_resale_index(train_df, resale_index_df)
eval_df = join_resale_index(eval_df, resale_index_df)
hold_df = join_resale_index(hold_df, resale_index_df)

# Verify the output
train_df[['month', 'quarter', 'resale_price_index']].head()


,month,quarter,resale_price_index
0,2022-03-01,2022 1Q,159.5
1,2022-04-01,2022 2Q,163.9
2,2022-04-01,2022 2Q,163.9
3,2022-04-01,2022 2Q,163.9
4,2022-05-01,2022 2Q,163.9


In [7]:
# flat_type_rank_map
# Calculate median resale price by flat_type
flat_type_medians = train_df.groupby('flat_type')['resale_price'].median().sort_values(ascending=False)
display(flat_type_medians)

# Creating an ordinal mapping based on price
flat_type_rank_map = {flat: i for i, flat in enumerate(flat_type_medians.index[::-1])}
print(flat_type_rank_map)

flat_type
MULTI-GENERATION    829000.0
EXECUTIVE           670000.0
5 ROOM              558000.0
4 ROOM              460000.0
3 ROOM              330000.0
2 ROOM              260000.0
1 ROOM              190000.0
Name: resale_price, dtype: float64

{'1 ROOM': 0, '2 ROOM': 1, '3 ROOM': 2, '4 ROOM': 3, '5 ROOM': 4, 'EXECUTIVE': 5, 'MULTI-GENERATION': 6}


In [8]:
# region_rank_map
# Calculate median resale price for each region in region_ura
region_medians = train_df.groupby('region_ura')['resale_price'].median().sort_values(ascending=False)
display(region_medians)

# Example mapping based on the calculated medians
region_rank_map = {region: i for i, region in enumerate(region_medians.index[::-1])}
# This gives a higher rank to regions with higher median prices
print(region_rank_map)

region_ura
CENTRAL REGION       560000.0
EAST REGION          480000.0
NORTH-EAST REGION    470000.0
WEST REGION          440000.0
NORTH REGION         412000.0
Name: resale_price, dtype: float64

{'NORTH REGION': 0, 'WEST REGION': 1, 'NORTH-EAST REGION': 2, 'EAST REGION': 3, 'CENTRAL REGION': 4}


In [9]:
# Calculate median resale price by town
town_medians = train_df.groupby('town')['resale_price'].median().sort_values(ascending=False)
display(town_medians)

# Create ordinal ranking mapping for town based on median resale_price
town_rank_map = {town: i for i, town in enumerate(town_medians.index[::-1])}

# Display the mapping
print("Town Rank Map (Lower Rank = Lower Median Price):")
display(town_rank_map)


town
BUKIT TIMAH        725000.0
BISHAN             659000.0
QUEENSTOWN         645000.0
BUKIT MERAH        620000.0
CENTRAL AREA       538000.0
PASIR RIS          530000.0
KALLANG/WHAMPOA    530000.0
TAMPINES           500000.0
SERANGOON          498000.0
PUNGGOL            497000.0
MARINE PARADE      479000.0
SENGKANG           475000.0
CLEMENTI           473500.0
HOUGANG            459888.0
BUKIT PANJANG      455000.0
CHOA CHU KANG      452000.0
SEMBAWANG          440000.0
TOA PAYOH          440000.0
JURONG WEST        430000.0
GEYLANG            425000.0
JURONG EAST        422000.0
WOODLANDS          420000.0
BUKIT BATOK        419444.0
BEDOK              400000.0
YISHUN             398000.0
ANG MO KIO         383000.0
Name: resale_price, dtype: float64

Town Rank Map (Lower Rank = Lower Median Price):


{'ANG MO KIO': 0,
 'YISHUN': 1,
 'BEDOK': 2,
 'BUKIT BATOK': 3,
 'WOODLANDS': 4,
 'JURONG EAST': 5,
 'GEYLANG': 6,
 'JURONG WEST': 7,
 'TOA PAYOH': 8,
 'SEMBAWANG': 9,
 'CHOA CHU KANG': 10,
 'BUKIT PANJANG': 11,
 'HOUGANG': 12,
 'CLEMENTI': 13,
 'SENGKANG': 14,
 'MARINE PARADE': 15,
 'PUNGGOL': 16,
 'SERANGOON': 17,
 'TAMPINES': 18,
 'KALLANG/WHAMPOA': 19,
 'PASIR RIS': 20,
 'CENTRAL AREA': 21,
 'BUKIT MERAH': 22,
 'QUEENSTOWN': 23,
 'BISHAN': 24,
 'BUKIT TIMAH': 25}

In [10]:
# Calculate median resale price by storey_range
storey_range_medians = train_df.groupby('storey_range')['resale_price'].median().sort_values(ascending=False)
display(storey_range_medians)

# Create ordinal ranking mapping for storey_range based on median resale_price
storey_range_rank_map = {storey_range: i for i, storey_range in enumerate(storey_range_medians.index[::-1])}

# Display the mapping
print("storey_range Rank Map (Lower Rank = Lower Median Price):")
display(storey_range_rank_map)

storey_range
49 TO 51    1130000.0
46 TO 48    1068400.0
43 TO 45    1011500.0
40 TO 42     920000.0
37 TO 39     880000.0
34 TO 36     868000.0
31 TO 33     840000.0
28 TO 30     820000.0
25 TO 27     728000.0
22 TO 24     650000.0
19 TO 21     625000.0
16 TO 18     540000.0
13 TO 15     505000.0
10 TO 12     463000.0
07 TO 09     450000.0
04 TO 06     435000.0
01 TO 03     415000.0
Name: resale_price, dtype: float64

storey_range Rank Map (Lower Rank = Lower Median Price):


{'01 TO 03': 0,
 '04 TO 06': 1,
 '07 TO 09': 2,
 '10 TO 12': 3,
 '13 TO 15': 4,
 '16 TO 18': 5,
 '19 TO 21': 6,
 '22 TO 24': 7,
 '25 TO 27': 8,
 '28 TO 30': 9,
 '31 TO 33': 10,
 '34 TO 36': 11,
 '37 TO 39': 12,
 '40 TO 42': 13,
 '43 TO 45': 14,
 '46 TO 48': 15,
 '49 TO 51': 16}

In [11]:
# Calculate median resale price by flat_model
flat_model_medians = train_df.groupby('flat_model')['resale_price'].median().sort_values(ascending=False)
display(flat_model_medians)

# Create ordinal ranking mapping for flat_model based on median resale_price
flat_model_rank_map = {flat_model: i for i, flat_model in enumerate(flat_model_medians.index[::-1])}

# Display the mapping
print("flat_model Rank Map (Lower Rank = Lower Median Price):")
display(flat_model_rank_map)


flat_model
Type S2                   1100000.0
Type S1                    970044.0
Premium Apartment Loft     920000.0
Terrace                    845000.0
Multi Generation           829000.0
Premium Maisonette         808000.0
Model A-Maisonette         762000.0
DBSS                       756500.0
3Gen                       722500.0
Adjoined flat              720000.0
Maisonette                 718000.0
Improved-Maisonette        698888.0
Apartment                  655000.0
Premium Apartment          510000.0
Improved                   488000.0
Model A                    450000.0
Model A2                   378000.0
Simplified                 365000.0
New Generation             350000.0
Standard                   330000.0
2-room                     305000.0
Name: resale_price, dtype: float64

flat_model Rank Map (Lower Rank = Lower Median Price):


{'2-room': 0,
 'Standard': 1,
 'New Generation': 2,
 'Simplified': 3,
 'Model A2': 4,
 'Model A': 5,
 'Improved': 6,
 'Premium Apartment': 7,
 'Apartment': 8,
 'Improved-Maisonette': 9,
 'Maisonette': 10,
 'Adjoined flat': 11,
 '3Gen': 12,
 'DBSS': 13,
 'Model A-Maisonette': 14,
 'Premium Maisonette': 15,
 'Multi Generation': 16,
 'Terrace': 17,
 'Premium Apartment Loft': 18,
 'Type S1': 19,
 'Type S2': 20}

In [12]:

# Apply the mapping to create a 'flat_type_rank' column
train_df['flat_type_rank']=train_df['flat_type'].map(flat_type_rank_map)
eval_df['flat_type_rank']=eval_df['flat_type'].map(flat_type_rank_map).fillna(0)
hold_df['flat_type_rank']=hold_df['flat_type'].map(flat_type_rank_map).fillna(0)


# Apply the mapping to create a 'town_region_ura_rank' column
train_df['region_ura_rank']=train_df['region_ura'].map(region_rank_map)
eval_df['region_ura_rank']=eval_df['region_ura'].map(region_rank_map).fillna(0)
hold_df['region_ura_rank']=hold_df['region_ura'].map(region_rank_map).fillna(0)


# Apply the mapping to create a 'town_rank' column
train_df['town_rank'] = train_df['town'].map(town_rank_map)
eval_df['town_rank'] = eval_df['town'].map(town_rank_map).fillna(0)
hold_df['town_rank'] = hold_df['town'].map(town_rank_map).fillna(0)

# Apply the mapping to create a 'storey_range_rank' column
train_df['storey_range_rank'] = train_df['storey_range'].map(storey_range_rank_map)
eval_df['storey_range_rank'] = eval_df['storey_range'].map(storey_range_rank_map).fillna(0)
hold_df['storey_range_rank'] = hold_df['storey_range'].map(storey_range_rank_map).fillna(0)


# Apply the mapping to create a 'flat_model_rank' column
train_df['flat_model_rank'] = train_df['flat_model'].map(flat_model_rank_map)
eval_df['flat_model_rank'] = eval_df['flat_model'].map(flat_model_rank_map).fillna(0)
hold_df['flat_model_rank'] = hold_df['flat_model'].map(flat_model_rank_map).fillna(0)

In [13]:
import json

# Define the paths for each map
flat_type_rank_map_path = processed_data_path / 'flat_type_rank_map.json'
region_rank_map_path = processed_data_path / 'region_rank_map.json'
town_rank_map_path = processed_data_path / 'town_rank_map.json'
storey_range_rank_map_path = processed_data_path / 'storey_range_rank_map.json'
flat_model_rank_map_path = processed_data_path / 'flat_model_rank_map.json'


# Save the maps to JSON files
with open(flat_type_rank_map_path, 'w') as f:
    json.dump(flat_type_rank_map, f, indent=4)
    
with open(region_rank_map_path, 'w') as f:
    json.dump(region_rank_map, f, indent=4)

with open(town_rank_map_path, 'w') as f:
    json.dump(town_rank_map, f, indent=4)

with open(storey_range_rank_map_path, 'w') as f:
    json.dump(storey_range_rank_map, f, indent=4)

with open(flat_model_rank_map_path, 'w') as f:
    json.dump(flat_model_rank_map, f, indent=4)

print(f"Encoding maps saved successfully to: {processed_data_path}")


Encoding maps saved successfully to: /Volumes/SANDISKUSBC/MLOps/hdb_price_prediction_mlops/data/processed


### Save featured_df

In [14]:


# Save the dataframes to CSV files
train_df.to_csv(processed_data_path/'featured_train.csv', index=False)
eval_df.to_csv(processed_data_path/'featured_eval.csv', index=False)
hold_df.to_csv(processed_data_path/'featured_hold.csv', index=False)

print(f"Dataframes saved successfully to: {processed_data_path}")


Dataframes saved successfully to: /Volumes/SANDISKUSBC/MLOps/hdb_price_prediction_mlops/data/processed
